<a href="https://colab.research.google.com/github/davidpassos123-arch/Pipeline_2_teste/blob/main/aula1/scripts/tokenizacao_stemming_ptbr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fundamentos de NLP: Tokenização, Stemming, BoW e Embeddings (PT-BR)

Este notebook é uma demonstração didática dos conceitos de NLP, evoluindo de técnicas básicas até o estado da arte de representação vetorial (Embeddings).

---

## 1. Configuração do Ambiente
Instalamos as bibliotecas necessárias para processar texto e carregar modelos de última geração.

In [1]:
# Instalação das bibliotecas necessárias
!pip install -q gensim nltk scikit-learn pandas numpy sentence-transformers FlagEmbedding

import nltk
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder
from FlagEmbedding import FlagModel

# Downloads de dados específicos do NLTK para Português
nltk.download('punkt')
nltk.download('rslp')
nltk.download('punkt_tab')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.6 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## 2. Tokenização

Divisão do texto em tokens (palavras ou sentenças).

In [2]:
texto = "O suspeito foi detido em flagrante. A Constituição garante o direito à defesa."
print("Sentenças:", sent_tokenize(texto, language='portuguese'))
print("Palavras:", word_tokenize(texto, language='portuguese')[:25])

Sentenças: ['O suspeito foi detido em flagrante.', 'A Constituição garante o direito à defesa.']
Palavras: ['O', 'suspeito', 'foi', 'detido', 'em', 'flagrante', '.', 'A', 'Constituição', 'garante', 'o', 'direito', 'à', 'defesa', '.']


## 3. Stemming

Redução de palavras ao radical (morfologia).

In [3]:
stemmer = RSLPStemmer()
for p in ["justiça", "jurídico", "juiz"]:
    print(f"{p} -> {stemmer.stem(p)}")

justiça -> justiç
jurídico -> juríd
juiz -> juiz


## 4. Bag-of-Words (BoW)

Representação por contagem. Note como a ordem não importa.

In [4]:
corpus = ["o juiz absolveu o réu", "o réu absolveu o juiz"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
display(pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), index=corpus))

,absolveu,juiz,réu
o juiz absolveu o réu,1,1,1
o réu absolveu o juiz,1,1,1


## 5. Embeddings Estáticos (Word2Vec, GloVe)

Vetores densos onde a proximidade geométrica indica similaridade.

In [5]:
try:
    model_glove = api.load("glove-wiki-gigaword-50")
    print("Similaridade entre 'judge' e 'court':", model_glove.similarity('judge', 'court'))
except: print("Erro no download.")

[==================================================] 100.0% 66.0/66.0MB downloaded
Similaridade entre 'judge' e 'court': 0.870952


## 6. Embeddings Contextuais (BERT, Sentence-Transformers e BGE-M3)

### O contexto muda tudo
Em 2018, o BERT representou uma ruptura: ao invés de um vetor fixo, cada palavra ganha representações diferentes dependendo da frase.

---

In [6]:
print("--- 1. Exemplo com Sentence-Transformers (Português) ---")
model_st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

frases = [
    "O réu foi condenado pelo magistrado.",
    "O juiz sentenciou o acusado.",
    "O policial efetuou a prisão em flagrante."
]

embeddings_st = model_st.encode(frases)

print(f"Similaridade (Juiz vs Sentença): {cosine_similarity([embeddings_st[0]], [embeddings_st[1]])[0][0]:.4f}")
print(f"Similaridade (Juiz vs Policial): {cosine_similarity([embeddings_st[0]], [embeddings_st[2]])[0][0]:.4f}")

--- 1. Exemplo com Sentence-Transformers (Português) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similaridade (Juiz vs Sentença): 0.8698
Similaridade (Juiz vs Policial): 0.5219


In [7]:
print("\n--- 2. Exemplo com BGE-M3 (Estado da Arte) ---")
try:
    model_bge = FlagModel('BAAI/bge-m3', use_fp16=True)
    embeddings_bge = model_bge.encode(frases)

    score_1 = np.dot(embeddings_bge[0], embeddings_bge[1])
    score_2 = np.dot(embeddings_bge[0], embeddings_bge[2])

    print(f"Similaridade BGE-M3 (Juiz vs Sentença): {score_1:.4f}")
    print(f"Similaridade BGE-M3 (Juiz vs Policial): {score_2:.4f}")
except Exception as e:
    print(f"Nota: BGE-M3 pode exigir GPU ou mais memória. Erro: {e}")


--- 2. Exemplo com BGE-M3 (Estado da Arte) ---


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Similaridade BGE-M3 (Juiz vs Sentença): 0.9037
Similaridade BGE-M3 (Juiz vs Policial): 0.6655


### 6.3 Desempenho: Por que Bi-Encoders são o padrão para RAG?

**Pergunta:** Por que usar BERT diretamente (Cross-Encoder) para busca em 100.000 documentos é inviável?

**Resposta:** O BERT original foi projetado como um **Cross-Encoder**: ele precisa de dois textos juntos para produzir um score. Se você tem 100.000 documentos, precisaria rodar o modelo 100.000 vezes para cada busca.

O **Sentence-Transformers (Bi-Encoder)** resolve isso: você faz o embedding dos 100.000 documentos **uma única vez** e armazena. Na busca, faz apenas **1 embedding** da pergunta e compara vetores em milissegundos.

In [8]:
from sentence_transformers import CrossEncoder
import time

print("--- Comparação de Velocidade: Cross-Encoder vs Bi-Encoder ---")
query = "Qual a pena para o crime de peculato?"
docs = [f"Documento jurídico número {i} contendo informações sobre leis penais..." for i in range(100)]

# 1. Simulação Cross-Encoder (Lento)
model_ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
start_ce = time.time()
# Cross-encoder precisa processar cada par (query, doc)
scores_ce = model_ce.predict([(query, doc) for doc in docs])
end_ce = time.time()
time_ce = end_ce - start_ce
print(f"Tempo Cross-Encoder (100 documentos): {time_ce:.4f} segundos")

# 2. Simulação Bi-Encoder (Rápido)
# Nota: Em um sistema real, os docs já estariam encodados.
doc_embeddings = model_st.encode(docs)

start_bi = time.time()
query_embedding = model_st.encode(query)
similarities = cosine_similarity([query_embedding], doc_embeddings)
end_bi = time.time()
time_bi = end_bi - start_bi
print(f"Tempo Bi-Encoder (100 documentos): {time_bi:.4f} segundos")

print(f"\nO Bi-Encoder foi aproximadamente {time_ce/time_bi:.1f}x mais rápido para apenas 100 docs.")
print("Imagine a diferença para 100.000 documentos!")

--- Comparação de Velocidade: Cross-Encoder vs Bi-Encoder ---


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Tempo Cross-Encoder (100 documentos): 1.7313 segundos
Tempo Bi-Encoder (100 documentos): 0.0849 segundos

O Bi-Encoder foi aproximadamente 20.4x mais rápido para apenas 100 docs.
Imagine a diferença para 100.000 documentos!


## 7. Por que usar em RAG?

A evolução de BoW -> Word2Vec -> BGE-M3 permitiu que sistemas de IA jurídica não apenas "contem palavras", mas entendam conceitos complexos, sinônimos e contextos específicos, tornando a recuperação de documentos em RAG muito mais precisa.